# Phase 1 — Electrical Feasibility: Distribution Grid Screening
**Ontario Distribution-Connected Solar Siting | 10+ MW Projects**

Stack:
- **PostGIS** — all spatial logic (dissolve, buffer, intersect, area, clip)
- **GeoPandas** — load results, reproject for visualization
- **Folium** — single combined interactive map at the end

**CRS convention**
- EPSG:3161 (NAD83/Ontario MNR Lambert) — all spatial calculations (buffer, area, intersect)
- EPSG:4326 (WGS84) — data exchange with GeoPandas and Folium

**Steps**
1. Build County AOI (dissolved + buffered township boundary)
2. Clip OEM grid to AOI (capacity > 10 MW)
3. Select lots intersecting OEM ≥ 10 MW zones

---
Change only `COUNTY_NAME` in Section 0 to reproduce for any county.

## Dependencies

In [56]:
%pip install geopandas folium branca sqlalchemy psycopg2-binary pyogrio

Note: you may need to restart the kernel to use updated packages.


## 0 · Configuration

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PROJECT PARAMETERS — edit this cell only to run for a different county
# ─────────────────────────────────────────────────────────────────────────────

import os

COUNTY_NAME    = "Peterborough"   # Must match adm_district_township.name_2
AOI_BUFFER_M   = 2000             # AOI buffer in metres (EPSG:3161)
MIN_POLYGON_HA = 30               # OEM polygon noise filter (hectares)

# PostGIS connection from .env
PG_CONN = os.environ["POSTGRES_CONNECTION_STRING"]

# Output schema — tables are named per step and county automatically
OUTPUT_SCHEMA = "analysis"

# CRS
CRS_WGS84   = 4326
CRS_NAD83   = 4269   # NAD83 geographic — bridge CRS to avoid PROJ grid dependency
CRS_ONTARIO = 5321   # NAD83 / Ontario MNR Lambert — all calculations (buffer, area, intersect)

## 1 · Environment Setup

In [2]:
import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import MiniMap
from branca.colormap import linear
from sqlalchemy import create_engine, text

print(f"GeoPandas : {gpd.__version__}")
print(f"Folium    : {folium.__version__}")

GeoPandas : 1.1.2
Folium    : 0.20.0


## 2 · PostGIS Connection

In [3]:
engine = create_engine(PG_CONN)

def read_postgis(sql: str, geom_col: str = "geom") -> gpd.GeoDataFrame:
    """Execute a PostGIS query and return a WGS84 GeoDataFrame."""
    gdf = gpd.read_postgis(sql, engine, geom_col=geom_col)
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=CRS_WGS84)
    return gdf.to_crs(epsg=CRS_WGS84)


def save_to_postgis(gdf: gpd.GeoDataFrame, table: str, label: str) -> None:
    """Write a GeoDataFrame to PostGIS in EPSG:3161 and create a GiST spatial index."""
    geom_col = gdf.geometry.name  # actual geometry column name (e.g. "geom")
    gdf.to_crs(epsg=CRS_ONTARIO).to_postgis(
        name=table,
        con=engine,
        schema=OUTPUT_SCHEMA,
        if_exists="replace",
        index=False,
        chunksize=500
    )
    with engine.connect() as conn:
        conn.execute(text(
            f"CREATE INDEX IF NOT EXISTS {table}_geom_idx "
            f"ON {OUTPUT_SCHEMA}.{table} USING GIST({geom_col})"
        ))
        conn.commit()
    print(f"  {label:35s} → {OUTPUT_SCHEMA}.{table} "
          f"({len(gdf):,} rows, EPSG:{CRS_ONTARIO}, GiST index)")


with engine.connect() as conn:
    print("PostGIS:", conn.execute(text("SELECT PostGIS_Full_Version()")).scalar())

PostGIS: POSTGIS="3.3.4 0" [EXTENSION] PGSQL="150" GEOS="3.11.2-CAPI-1.17.2" SFCGAL="SFCGAL 1.4.1, CGAL 5.5.2, BOOST 1.82.0" PROJ="9.2.1" GDAL="GDAL 3.6.4, released 2023/04/17" LIBXML="2.11.4" LIBJSON="0.16" LIBPROTOBUF="1.4.1" WAGYU="0.5.0 (Internal)" TOPOLOGY RASTER


---
## Step 1 — Build County AOI

PostGIS CTE chain:
1. Keep townships whose centroid intersects the county admin boundary
2. Dissolve into a single polygon
3. Buffer by 2 km in EPSG:3161, return in WGS84

In [4]:
gdf_aoi = read_postgis(f"""
WITH filtered AS (
    SELECT t.geom
    FROM administrative.ontario_geographic_township t
    JOIN administrative.adm_district_township d
      ON ST_Intersects(
             ST_PointOnSurface(ST_Transform(t.geom, {CRS_WGS84})),
             d.geom
         )
    WHERE d.name_2 = '{COUNTY_NAME}'
),
dissolved AS (
    SELECT ST_Union(geom) AS geom FROM filtered
),
buffered AS (
    SELECT ST_Buffer(ST_Transform(geom, {CRS_ONTARIO}), {AOI_BUFFER_M}) AS geom
    FROM dissolved
)
SELECT ST_Transform(geom, {CRS_WGS84}) AS geom FROM buffered
""")

print(f"AOI: {len(gdf_aoi)} polygon(s) for '{COUNTY_NAME}'")
center = [gdf_aoi.geometry.centroid.y.mean(), gdf_aoi.geometry.centroid.x.mean()]
print(f"Centroid: {center[0]:.4f}N, {center[1]:.4f}W")

county_slug = COUNTY_NAME.lower().replace(" ", "_")
save_to_postgis(gdf_aoi, f"aoi_county_{county_slug}", "Step 1 — County AOI")

AOI: 1 polygon(s) for 'Simcoe'
Centroid: 44.4767N, -79.7328W
  Step 1 — County AOI                 → analysis.aoi_county_simcoe (1 rows, EPSG:5321, GiST index)


In [5]:
# Reference layers for map context (loaded once, used in combined map)
gdf_townships = read_postgis(
    "SELECT geom FROM administrative.ontario_geographic_township"
)
gdf_adm = read_postgis(
    f"SELECT name_2, geom FROM administrative.adm_district_township "
    f"WHERE name_2 = '{COUNTY_NAME}'"
)
print(f"Townships: {len(gdf_townships):,} | Admin boundary: {len(gdf_adm)} feature(s)")

Townships: 2,528 | Admin boundary: 20 feature(s)


---
## Step 2 — Clip OEM Grid to AOI (capacity > 10 MW)

AOI is read from `analysis.aoi_county_{slug}` (saved by Step 1).
Filters OEM polygons with `capacityrange = '> 10 MW'` and area > 30 ha,
then clips them to the county AOI. All spatial operations in EPSG:5321.

In [6]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

gdf_oem_county = read_postgis(f"""
WITH aoi AS (
    SELECT geom FROM {OUTPUT_SCHEMA}.aoi_county_{county_slug}
),
oem_clipped AS (
    SELECT
        g.capacityrange,
        g.ldc_name,
        g.station_name,
        g.feeder_name,
        g.capacity,
        g.feeder_ltl_voltage_3ph AS voltage_3ph,
        ST_Intersection(ST_Transform(g.geom, {CRS_ONTARIO}), a.geom) AS geom
    FROM grid.oem_grid g
    JOIN aoi a ON ST_Intersects(ST_Transform(g.geom, {CRS_ONTARIO}), a.geom)
    WHERE g.capacity IS NOT NULL
      AND g.capacityrange = '> 10 MW'
),
oem_dumped AS (
    SELECT
        capacityrange, ldc_name, station_name, feeder_name,
        capacity, voltage_3ph,
        (ST_Dump(geom)).geom AS geom
    FROM oem_clipped
),
oem_filtered AS (
    SELECT
        capacityrange, ldc_name, station_name, feeder_name,
        capacity, voltage_3ph,
        ST_Area(ST_MakeValid(geom)) / 10000 AS hectarea,
        ST_MakeValid(geom) AS geom
    FROM oem_dumped
    WHERE ST_Area(ST_MakeValid(geom)) / 10000 > {MIN_POLYGON_HA}
)
SELECT
    capacityrange, ldc_name, station_name, feeder_name,
    capacity, voltage_3ph, hectarea,
    ST_Transform(geom, {CRS_WGS84}) AS geom
FROM oem_filtered
""")

print(f"OEM polygons >10 MW clipped to AOI: {len(gdf_oem_county):,}")
print()
print(gdf_oem_county[["station_name", "feeder_name", "capacity", "voltage_3ph"]]
      .sort_values("capacity", ascending=False)
      .drop_duplicates()
      .to_string(index=False))

save_to_postgis(gdf_oem_county, f"oem_county_{county_slug}", "Step 2 — OEM grid (>10 MW)")

OEM polygons >10 MW clipped to AOI: 162

        station_name feeder_name  capacity  voltage_3ph
         ALLISTON TS          M3      30.0         44.0
         ALLISTON TS          M6      30.0         44.0
       Midhurst T.S.       23M24      30.0         44.0
       Midhurst T.S.       23M28      30.0         44.0
         Alliston TS          M2      30.0         44.0
          HOLLAND TS         M10      30.0         44.0
         Barrie T.S.        13M6      30.0         44.0
         Barrie T.S.        13M2      30.0         44.0
   MIDHURST TS DESN1          M6      30.0         44.0
       Midhurst T.S.       23M27      30.0         44.0
   MIDHURST TS DESN1          M7      30.0         44.0
       Midhurst T.S.       23M22      30.0         44.0
       Midhurst T.S.        23M6      30.0         44.0
       Midhurst T.S.        23M7      30.0         44.0
        Holland T.S.      153M10      30.0         44.0
         ALLISTON TS          M2      30.0         44.0
       

---
## Step 3 — Select Lots Intersecting OEM ≥ 10 MW

OEM capacity zones are read from `analysis.oem_county_{slug}` (saved by Step 2).
Spatial join between cadastral lots and OEM zones. The smaller OEM set is
transformed to match the lots' native CRS so PostGIS can use the GiST index on
`cadastre.ontario_lots.geom`. Grid attributes are aggregated with
`STRING_AGG(DISTINCT ...)` to concatenate all distinct values when a lot
intersects multiple OEM features. No area filter — all intersecting lots retained.

In [7]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

gdf_lots_result = read_postgis(f"""
WITH lots_intersecting AS (
    SELECT
        a.lot_ident,
        a.concession_ident,
        a.geographic_township_name,
        (ST_Area(a.geom) / 4046.856) AS acre_total,
        b.capacity,
        b.voltage_3ph::text AS voltage_3ph,
        b.station_name,
        b.feeder_name,
        b.ldc_name,
        ST_Transform(a.geom, {CRS_WGS84}) AS geom
    FROM cadastre.ontario_lots AS a
    JOIN {OUTPUT_SCHEMA}.oem_county_{county_slug} AS b
      ON ST_Intersects(ST_Transform(a.geom, {CRS_ONTARIO}), b.geom)
     AND ST_Area(
           ST_Intersection(ST_Transform(a.geom, {CRS_ONTARIO}), b.geom)
         ) > 0
    WHERE a.concession_ident <> ' '
)
SELECT
    lot_ident,
    concession_ident,
    geographic_township_name,
    ROUND(acre_total::numeric, 2) AS acre_total,
    STRING_AGG(DISTINCT capacity::text, ', ' ORDER BY capacity::text)    AS grid_capacity_mw,
    STRING_AGG(DISTINCT voltage_3ph, ', ' ORDER BY voltage_3ph)          AS voltage_3ph,
    STRING_AGG(DISTINCT station_name, ', ' ORDER BY station_name)        AS station_name,
    STRING_AGG(DISTINCT feeder_name, ', ' ORDER BY feeder_name)          AS feeder_name,
    STRING_AGG(DISTINCT ldc_name, ', ' ORDER BY ldc_name)                AS ldc_name,
    geom
FROM lots_intersecting
GROUP BY
    lot_ident,
    concession_ident,
    geographic_township_name,
    acre_total,
    geom
ORDER BY acre_total DESC
""")

print(f"Selected lots: {len(gdf_lots_result):,}")
print(f"Area range   : {gdf_lots_result['acre_total'].min():.1f} – {gdf_lots_result['acre_total'].max():.1f} ac")
print("\nGrid capacity distribution:")
print(gdf_lots_result["grid_capacity_mw"].value_counts().sort_index(ascending=False).to_string())
print("\nLots per township:")
print(gdf_lots_result["geographic_township_name"].value_counts().to_string())

save_to_postgis(gdf_lots_result, f"selected_lots_{county_slug}", "Step 3 — Selected lots")

Selected lots: 2,516
Area range   : 0.0 – 1606.0 ac

Grid capacity distribution:
grid_capacity_mw
30                  202
29.9, 30             17
29.9                175
29.6, 30              3
29.6                 32
29.4                 20
29.3, 29.4            9
29.3                 90
29.2                 53
29.1, 29.2            4
29.1                  3
28.8, 30             12
28.8, 29.9            3
28.8                162
28.7, 30              5
28.7, 29.9            4
28.7, 29.6            5
28.7                 31
28.4, 29.6, 30        1
28.4, 29.6           10
28.4                 64
28.2, 30              4
28.2, 29.3            1
28.2, 28.8            6
28.2                173
28, 29.1, 29.2        1
28, 29.1              1
28, 28.4              5
28                   44
27.5, 30              5
27.5, 28.8, 30        1
27.5, 28.8            2
27.5, 28.2, 30        1
27.5, 28.2            2
27.5                 11
26.9, 29.4            1
26.9, 29.3, 29.4      2
26.9, 29.3    

In [8]:
print(gdf_lots_result[
    ["lot_ident", "concession_ident", "geographic_township_name",
     "acre_total", "grid_capacity_mw", "voltage_3ph", "station_name", "feeder_name"]
].head(20).to_string(index=False))

lot_ident           concession_ident geographic_township_name  acre_total grid_capacity_mw voltage_3ph                     station_name              feeder_name
          MILITARY AND NAVAL RESERVE                      TAY     1606.03       10.4, 10.5          44 WAUBAUSHENE TS, Waubaushene T.S.                 98M3, M3
   LOT 10                     CON 16                SUNNIDALE      884.92             16.9          44                       Stayner TS STAYNER 2M4, STAYNER 2M5
          MILITARY AND NAVAL RESERVE                     TINY      768.58             10.4          44                 Waubaushene T.S.                     98M3
    LOT 9                     CON 16                SUNNIDALE      749.28             16.9          44                       Stayner TS              STAYNER 2M4
    LOT 8                     CON 16                SUNNIDALE      608.69             16.9          44                       Stayner TS              STAYNER 2M4
   LOT 25                      CON

---
## Combined Map — All Layers

Single Folium map with AOI outline, OEM capacity zones, selected lots, and county admin boundary.
Saved to `map_phase1_combined.html` (not displayed inline — open the HTML file in a browser).

In [9]:
# ── Combined map: AOI + OEM grid + selected lots ──────────────────────────────
from branca.colormap import linear

county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Colormaps
cap_vals = gdf_oem_county["capacity"].dropna()
cap_cmap = linear.YlGn_09.scale(cap_vals.min(), cap_vals.max())
cap_cmap.caption = "OEM Capacity (MW)"

acre_vals = gdf_lots_result["acre_total"].dropna()
acre_cmap = linear.OrRd_09.scale(acre_vals.min(), acre_vals.max())
acre_cmap.caption = "Lot Area (acres)"

m = folium.Map(location=center, zoom_start=10, tiles="CartoDB positron")
MiniMap(toggle_display=True).add_to(m)

# Layer 1 — County admin boundary (reference)
folium.GeoJson(
    gdf_adm.__geo_interface__,
    name="County admin boundary",
    style_function=lambda _: {
        "fillColor": "none", "color": "#e67e22", "weight": 2.5, "dashArray": "6 4"
    }
).add_to(m)

# Layer 2 — AOI outline
folium.GeoJson(
    gdf_aoi.__geo_interface__,
    name=f"AOI — {COUNTY_NAME} (+{AOI_BUFFER_M // 1000} km buffer)",
    style_function=lambda _: {
        "fillColor": "#2e6b47", "color": "#1a3c2d", "weight": 2.5, "fillOpacity": 0.15
    }
).add_to(m)

# Layer 3 — OEM grid capacity zones
folium.GeoJson(
    gdf_oem_county.__geo_interface__,
    name="OEM grid (>10 MW)",
    style_function=lambda feat: {
        "fillColor": cap_cmap(feat["properties"].get("capacity") or 0),
        "color": "#2e6b47", "weight": 0.6, "fillOpacity": 0.65
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["station_name", "feeder_name", "capacity", "voltage_3ph", "hectarea"],
        aliases=["Station", "Feeder", "Capacity (MW)", "Voltage (kV)", "Area (ha)"],
        localize=True
    )
).add_to(m)

# Layer 4 — Selected lots
folium.GeoJson(
    gdf_lots_result.__geo_interface__,
    name="Selected lots",
    style_function=lambda feat: {
        "fillColor": acre_cmap(feat["properties"].get("acre_total") or 0),
        "color": "#7b3f00", "weight": 0.8, "fillOpacity": 0.75
    },
    highlight_function=lambda _: {
        "color": "#1a3c2d", "weight": 2.5, "fillOpacity": 0.9
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["lot_ident", "concession_ident", "geographic_township_name",
                "acre_total", "grid_capacity_mw", "voltage_3ph", "station_name"],
        aliases=["Lot", "Concession", "Township",
                 "Area (ac)", "Grid (MW)", "Voltage (kV)", "Station"],
        localize=True, sticky=True
    )
).add_to(m)

cap_cmap.add_to(m)
acre_cmap.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

out_html = f"map_phase1_{county_slug}.html"
m.save(out_html)
print(f"Saved: {out_html}")

Saved: map_phase1_simcoe.html


---
## 3 · Write Results to PostGIS

In [10]:
# Results already saved to analysis schema after each step.
county_slug = COUNTY_NAME.lower().replace(" ", "_")
print("Phase 1 tables in analysis schema:")
with engine.connect() as conn:
    rows = conn.execute(text(f"""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = '{OUTPUT_SCHEMA}'
          AND table_name LIKE '%{county_slug}%'
        ORDER BY table_name
    """)).fetchall()
    for r in rows:
        print(f"  {OUTPUT_SCHEMA}.{r[0]}")

Phase 1 tables in analysis schema:
  analysis.aoi_county_simcoe
  analysis.oem_county_simcoe
  analysis.selected_lots_simcoe


---
## Appendix — Reproducing for a Different County

Change `COUNTY_NAME` in Section 0 and run all cells. Output tables are county-scoped and never overwrite each other.

Each step **saves immediately** to the `analysis` schema via `save_to_postgis()`,
and each subsequent step **reads its inputs** from the previous step's saved table —
no CTE chains are re-derived.

```
Step 1  PostGIS query → gdf_aoi
        save_to_postgis() → analysis.aoi_county_<county>  (EPSG:5321 + GiST)
            │
            ▼  reads analysis.aoi_county_<county>
Step 2  PostGIS query → gdf_oem_county
        save_to_postgis() → analysis.oem_county_<county>  (EPSG:5321 + GiST)
            │
            ▼  reads analysis.oem_county_<county>
Step 3  PostGIS query → gdf_lots_result
        save_to_postgis() → analysis.selected_lots_<county>  (EPSG:5321 + GiST)
```

**Next phase** — Phase 2 takes `analysis.selected_lots_<county>` as input and carves out REA environmental
exclusion zones (wetlands, woodlands, ANSI, water + 120 m buffers).